In [1]:
!pip install datasets==2.19.0 awswrangler -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.0/374.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
import os
import pandas as pd
from datasets import load_dataset
import itertools
import time
import boto3

# ==============================================================================
# 1. CẤU HÌNH CLOUDFLARE R2 BẰNG BOTO3
# ==============================================================================
CLOUDFLARE_ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2" 
R2_ACCESS_KEY_ID = "352e66be1b3cb7fad1866c153222faf2"            
R2_SECRET_ACCESS_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"        
BUCKET_NAME = "amazon-reviews-test" 

ENDPOINT_URL = f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com"

# Khởi tạo s3_client thuần túy của AWS để giao tiếp với R2
s3_client = boto3.client(
    's3',
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto"
)

# ==============================================================================
# 2. BẮT ĐẦU CHƯƠNG TRÌNH
# ==============================================================================
# Cập nhật danh sách target: Electronics (để tải nốt phần review) và top 4 tiếp theo
TARGET_CATEGORIES = [
    "Electronics",
    "Tools_and_Home_Improvement",
    "Health_and_Household",
    "Kindle_Store",
    "Beauty_and_Personal_Care"
]

DATA_TYPES = ["review", "meta"]
CHUNK_SIZE = 100000 

def get_chunks(iterable, batch_size):
    iterator = iter(iterable)
    for first in iterator:
        yield itertools.chain([first], itertools.islice(iterator, batch_size - 1))

print("🚀 BẮT ĐẦU TẢI TIẾP DỮ LIỆU LÊN CLOUDFLARE R2 🚀")
print("-" * 75)

for category in TARGET_CATEGORIES:
    for data_type in DATA_TYPES:
        
        # LOGIC XỬ LÝ RIÊNG CHO ELECTRONICS: Chỉ tải phần review (vì code cũ đã bỏ qua review)
        if category == "Electronics" and data_type == "meta":
            print(f"\n⏭️ Đã bỏ qua: Ngành hàng [{category}] - Loại dữ liệu [{data_type.upper()}] (Giả định đã tải trước đó).")
            continue
        
        print(f"\n📦 Đang xử lý: Ngành hàng [{category}] - Loại dữ liệu [{data_type.upper()}]")
        start_time = time.time()
        
        try:
            hf_dataset_name = f"raw_{data_type}_{category}"
            dataset = load_dataset(
                "McAuley-Lab/Amazon-Reviews-2023", 
                hf_dataset_name, 
                split="full", 
                streaming=True,
                trust_remote_code=True
            )
            
            chunk_idx = 1
            
            for chunk in get_chunks(dataset, CHUNK_SIZE):
                chunk_data = list(chunk)
                df = pd.DataFrame(chunk_data).astype(str) 
                
                # Định nghĩa tên file: part_0001.parquet, part_0002.parquet...
                file_name = f"part_{chunk_idx:04d}.parquet" 
                
                # 1. Đường dẫn tạm ở máy ảo
                local_tmp_path = f"/tmp/{file_name}" 
                
                # 2. Đường dẫn đích trên Cloudflare R2
                s3_key = f"parquet-data/{data_type}/{category}/{file_name}"
                
                # BƯỚC 1: Lưu df ra file parquet tạm thời ở máy ảo
                df.to_parquet(local_tmp_path, index=False)
                
                # BƯỚC 2: Upload file tạm đó lên Cloudflare R2
                s3_client.upload_file(local_tmp_path, BUCKET_NAME, s3_key)
                
                # BƯỚC 3: Xóa file tạm để tránh đầy bộ nhớ
                os.remove(local_tmp_path)
                
                print(f"   -> Đã đẩy thành công {file_name} ({len(chunk_data)} dòng)")
                chunk_idx += 1
                
            elapsed_time = round(time.time() - start_time, 2)
            print(f"✅ Hoàn thành {data_type.upper()} của {category} trong {elapsed_time} giây!")
            
        except Exception as e:
            print(f"❌ Lỗi khi xử lý {data_type} của {category}: {e}")
            break 

print("\n🎉 XONG! TẤT CẢ DỮ LIỆU ĐÃ NẰM TRÊN CLOUDFLARE R2!")

🚀 BẮT ĐẦU TẢI TIẾP DỮ LIỆU LÊN CLOUDFLARE R2 🚀
---------------------------------------------------------------------------

📦 Đang xử lý: Ngành hàng [Electronics] - Loại dữ liệu [REVIEW]


   -> Đã đẩy thành công part_0001.parquet (100000 dòng)
   -> Đã đẩy thành công part_0002.parquet (100000 dòng)
   -> Đã đẩy thành công part_0003.parquet (100000 dòng)
   -> Đã đẩy thành công part_0004.parquet (100000 dòng)
   -> Đã đẩy thành công part_0005.parquet (100000 dòng)
   -> Đã đẩy thành công part_0006.parquet (100000 dòng)
   -> Đã đẩy thành công part_0007.parquet (100000 dòng)
   -> Đã đẩy thành công part_0008.parquet (100000 dòng)
   -> Đã đẩy thành công part_0009.parquet (100000 dòng)
   -> Đã đẩy thành công part_0010.parquet (100000 dòng)
   -> Đã đẩy thành công part_0011.parquet (100000 dòng)
   -> Đã đẩy thành công part_0012.parquet (100000 dòng)
   -> Đã đẩy thành công part_0013.parquet (100000 dòng)
   -> Đã đẩy thành công part_0014.parquet (100000 dòng)
   -> Đã đẩy thành công part_0015.parquet (100000 dòng)
   -> Đã đẩy thành công part_0016.parquet (100000 dòng)
   -> Đã đẩy thành công part_0017.parquet (100000 dòng)
   -> Đã đẩy thành công part_0018.parquet (10000

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
Retrying in 1s [Retry 1/5].


   -> Đã đẩy thành công part_0132.parquet (100000 dòng)


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
Retrying in 1s [Retry 1/5].


...

   -> Đã đẩy thành công part_0089.parquet (100000 dòng)


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Kindle_Store.jsonl
Retrying in 1s [Retry 1/5].


   -> Đã đẩy thành công part_0090.parquet (100000 dòng)
   -> Đã đẩy thành công part_0091.parquet (100000 dòng)


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Kindle_Store.jsonl
Retrying in 1s [Retry 1/5].


   -> Đã đẩy thành công part_0092.parquet (100000 dòng)
   -> Đã đẩy thành công part_0093.parquet (100000 dòng)
   -> Đã đẩy thành công part_0094.parquet (100000 dòng)
   -> Đã đẩy thành công part_0095.parquet (100000 dòng)
   -> Đã đẩy thành công part_0096.parquet (100000 dòng)
   -> Đã đẩy thành công part_0097.parquet (100000 dòng)
   -> Đã đẩy thành công part_0098.parquet (100000 dòng)
   -> Đã đẩy thành công part_0099.parquet (100000 dòng)
   -> Đã đẩy thành công part_0100.parquet (100000 dòng)
   -> Đã đẩy thành công part_0101.parquet (100000 dòng)
   -> Đã đẩy thành công part_0102.parquet (100000 dòng)
   -> Đã đẩy thành công part_0103.parquet (100000 dòng)
   -> Đã đẩy thành công part_0104.parquet (100000 dòng)
   -> Đã đẩy thành công part_0105.parquet (100000 dòng)
   -> Đã đẩy thành công part_0106.parquet (100000 dòng)
   -> Đã đẩy thành công part_0107.parquet (100000 dòng)
   -> Đã đẩy thành công part_0108.parquet (100000 dòng)
   -> Đã đẩy thành công part_0109.parquet (10000